# Context reranker v2 Colab workflow

This notebook prepares and smoke-tests the audit-first `context_reranker_v2` training workflow. The smoke run uses a tiny encoder and sampled rows only; it proves the dependency, path, checkpoint, eval, and offline prediction chain, not final model quality.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BRANCH = 'prepare-context-reranker-v2-colab'
REPO_URL = 'https://github.com/3516027002att-ui/Golf-Input-Method.git'
REPO_DIR = '/content/Golf-Input-Method'

!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"
%cd /content/Golf-Input-Method
!git fetch origin "$BRANCH:$BRANCH"
!git checkout "$BRANCH"
!git status --short --branch

In [ ]:
!python -m pip install -q -r requirements-train.txt

In [ ]:
from pathlib import Path
import shutil

DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/golf-ime-data-rebuild/clean_dataset_v3/left_context_only')
LOCAL_DATA_ROOT = Path('/content/golf-ime-data/context_v2')
RUN_DIR = Path('/content/drive/MyDrive/golf-ime-runs/context_reranker_v2_new_corpus')
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

source_files = {
    'train': DRIVE_DATA_ROOT / 'train_new_corpus.jsonl',
    'val': DRIVE_DATA_ROOT / 'val_new_corpus_v2.jsonl',
    'test': DRIVE_DATA_ROOT / 'test_new_corpus_v2.jsonl',
}
local_files = {}
for split, src in source_files.items():
    if not src.is_file():
        raise FileNotFoundError(src)
    dst = LOCAL_DATA_ROOT / src.name
    shutil.copy2(src, dst)
    local_files[split] = dst
    print(split, dst, dst.stat().st_size)


In [ ]:
AUDIT_REPORT = RUN_DIR / 'context_v2_new_corpus_audit.md'
!python training/context_reranker_v2.py audit-splits \
  --train "{local_files['train']}" \
  --val "{local_files['val']}" \
  --test "{local_files['test']}" \
  --report "$AUDIT_REPORT"
print(AUDIT_REPORT)

In [ ]:
def copy_head(src: Path, dst: Path, limit: int) -> None:
    with src.open('r', encoding='utf-8-sig') as reader, dst.open('w', encoding='utf-8', newline='\n') as writer:
        for index, line in enumerate(reader):
            if index >= limit:
                break
            if line.strip():
                writer.write(line.rstrip('\n') + '\n')

SMOKE_DIR = Path('/content/golf-ime-data/context_v2_smoke')
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_TRAIN = SMOKE_DIR / 'train_smoke.jsonl'
SMOKE_VAL = SMOKE_DIR / 'val_smoke.jsonl'
SMOKE_TEST = SMOKE_DIR / 'test_smoke.jsonl'
copy_head(local_files['train'], SMOKE_TRAIN, 64)
copy_head(local_files['val'], SMOKE_VAL, 24)
copy_head(local_files['test'], SMOKE_TEST, 24)
print(SMOKE_TRAIN, SMOKE_VAL, SMOKE_TEST)

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SMOKE_CHECKPOINT = RUN_DIR / 'smoke_checkpoint'
SMOKE_ENCODER = 'hf-internal-testing/tiny-random-bert'

!python training/context_reranker_v2.py train \
  --train "$SMOKE_TRAIN" \
  --val "$SMOKE_VAL" \
  --output-dir "$SMOKE_CHECKPOINT" \
  --encoder "$SMOKE_ENCODER" \
  --epochs 1 \
  --batch-size 2 \
  --eval-batch-size 4 \
  --context-mode online \
  --log-every 1 \
  --device "$DEVICE"
print(SMOKE_CHECKPOINT)

In [ ]:
EVAL_JSON = RUN_DIR / 'smoke_eval.json'
!python training/context_reranker_v2.py eval \
  --data "$SMOKE_TEST" \
  --checkpoint "$SMOKE_CHECKPOINT" \
  --context-mode online \
  --device "$DEVICE" > "$EVAL_JSON"
print(EVAL_JSON)

In [ ]:
!python scripts/predict_context_reranker_v2.py \
  --checkpoint "$SMOKE_CHECKPOINT" \
  --context-before "今天我想" \
  --composing "nihao" \
  --candidate "你好" \
  --candidate "拟好" \
  --candidate "泥嚎" \
  --device "$DEVICE"